<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Statistical-Embeddings/blob/main/StatisticalEmbeddingsSIM_Version2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

"""
Section 3.1 simulation + figures
Jointly trainable source-embedding version

Figure 1 revision:
    figure_source_complexity_logloss.png now contains
    ONLY two panels:
        1. Low-rank source effects, n_s = 40
        2. Low-rank source effects, n_s = 200

The unrestricted source-effect experiments remain
available in the saved CSV but are not shown
in Figure 1.

The code also addresses the identifiability
issue in Simulation 1 and Reviewer 1,
Comment 4, concerning source-effect recovery.

Google Drive checkpoints are saved every
600 seconds and completed Monte Carlo
replications can be resumed.

Run in Google Colab for automatic Drive mounting.

Outside Colab, mount or sync Google Drive and set:
    GOOGLE_DRIVE_CHECKPOINT_DIR

Outputs:
    source_complexity_logloss_results.csv
    source_complexity_embedding_dim_results.csv
    section_3_1_classification_results.csv
    section_3_1_covariance_results.csv
    section_3_1_ovb_results.csv
    section_3_1_ovb_source_effect_summary.csv
    section_3_1_ovb_source_effect_by_source.csv

Figures:
    figure_source_complexity_logloss.png
    figure_source_complexity_embedding_dim.png
    figure1.png
    figure2.png
    figure3.png
    figure4.png
    figure5.png
    figure6.png
    figure7_source_effect_recovery.png

NOTE:
The original uploaded script did not contain
the earlier ClassificationDGP implementation.
The reconstructed classification DGP below
should be verified against the original if
exact reproduction of Figures 2 and 3
is required.
"""

# ============================================================
# Imports
# ============================================================

from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path

import os
import copy
import random
import time
import pickle
import shutil
import threading
import hashlib
import json
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss


# ============================================================
# Global settings
# ============================================================

OUTDIR = "figures"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Final manuscript settings
N_REPS = 100

C = 8
P = 6
R0 = 4

GAMMA = 1.0
SIGMA_BETA = 0.5

K_GRID = [10, 25, 50, 100]
M_GRID = [2, 4, 8, 16]
NS_GRID = [40, 200]

# Both regimes are still simulated and saved.
# Figure 1 displays only the low-rank regime.
REGIMES = ["low_rank", "unrestricted"]

N_TEST = 20000

LAMBDA_GRID = [
    1e-4,
    1e-3,
    1e-2,
    1e-1
]

EPOCHS = 100
LR = 3e-2
PATIENCE = 20
VAL_FRAC = 0.2

TORCH_DTYPE = torch.float32


# ============================================================
# Utilities
# ============================================================

def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def set_all_seeds(seed: int) -> None:
    random.seed(int(seed))
    set_seed(int(seed))

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)


def sample_categorical(prob, rng):
    u = rng.uniform(size=prob.shape[0])
    return (
        prob.cumsum(axis=1) < u[:, None]
    ).sum(axis=1)


def train_val_split(n, val_frac, rng):
    idx = np.arange(n)
    rng.shuffle(idx)

    n_val = int(np.floor(val_frac * n))

    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]

    return tr_idx, val_idx


def pooled_within_class_covariance(
    H: np.ndarray,
    Y: np.ndarray
) -> np.ndarray:
    """
    Weighted average empirical covariance
    of H within each class (ddof=0).
    """

    H = np.asarray(H, dtype=float)
    Y = np.asarray(Y)

    n, p = H.shape
    cov = np.zeros((p, p))

    for c in np.unique(Y):
        h = H[Y == c]
        centered = h - h.mean(axis=0)
        cov += centered.T @ centered / n

    return cov


def summarize_with_ci(
    df: pd.DataFrame,
    group_col: str,
    metrics: List[str]
):
    """
    Monte Carlo means, standard errors,
    and 95% normal confidence interval
    half-widths.
    """

    pieces = []

    for group_value, group in df.groupby(
        group_col,
        sort=True
    ):

        rec = {
            group_col: group_value
        }

        for metric in metrics:

            arr = group[metric].to_numpy(
                dtype=float
            )

            rec[f"{metric}_mean"] = float(
                np.mean(arr)
            )

            rec[f"{metric}_ci"] = (
                float(
                    1.96 * np.std(
                        arr,
                        ddof=1
                    ) / np.sqrt(len(arr))
                )
                if len(arr) > 1
                else 0.0
            )

            rec[f"{metric}_se"] = (
                float(
                    np.std(
                        arr,
                        ddof=1
                    ) / np.sqrt(len(arr))
                )
                if len(arr) > 1
                else 0.0
            )

            rec[f"{metric}_n"] = len(arr)

        pieces.append(rec)

    return pd.DataFrame(pieces)


def canonical_multiclass_source_effects(
    Q: np.ndarray
) -> np.ndarray:
    """
    Double-center K-by-C softmax source logits.

    Row-centering removes source-specific shifts
    shared across all C classes.

    Column-centering removes source-average
    class offsets absorbed by alpha.

    This defines identifiable contrasts.
    """

    Q = np.asarray(Q, dtype=float)

    assert Q.ndim == 2

    return (
        Q
        - Q.mean(axis=0, keepdims=True)
        - Q.mean(axis=1, keepdims=True)
        + Q.mean()
    )


# ============================================================
# Classification DGP support
# ============================================================

@dataclass
class ClassificationDGP:

    n: int = 4000
    p: int = 6
    K: int = 5
    C: int = 3
    m: int = 2
    delta: float = 1.0
    seed: int = 1000


def generate_source_aware_classification(
    dgp: ClassificationDGP
):
    """
    Multiclass softmax DGP.

    Verify this reconstructed generator against
    the original implementation if exact
    reproduction of Figures 2 and 3 is needed.
    """

    rng = np.random.default_rng(dgp.seed)

    S = rng.integers(
        0,
        dgp.K,
        size=dgp.n
    )

    X = rng.normal(
        size=(dgp.n, dgp.p)
    )

    beta = rng.normal(
        scale=0.5,
        size=(dgp.C, dgp.p)
    )

    E = rng.normal(
        size=(dgp.K, dgp.m)
    )

    A = rng.normal(
        scale=1 / np.sqrt(dgp.m),
        size=(dgp.m, dgp.C)
    )

    Q = dgp.delta * E @ A

    Q = Q - Q.mean(
        axis=0,
        keepdims=True
    )

    logits = X @ beta.T + Q[S]

    Y = sample_categorical(
        softmax_np(logits),
        rng
    )

    return (
        X.astype(np.float32),
        S.astype(np.int64),
        Y.astype(np.int64)
    )


# ============================================================
# Google Drive checkpoints
# ============================================================

CHECKPOINT_INTERVAL_SECONDS = 600

EXPERIMENT_SCHEMA_VERSION = 3


class CheckpointManager:
    """
    Atomic periodic checkpoints.

    Each task is committed only after all
    of its models finish.

    An interruption during a replication
    causes only that unfinished replication
    to be rerun.
    """

    def __init__(
        self,
        checkpoint_dir: Path,
        config: dict,
        interval: float = 600
    ):

        self.dir = Path(checkpoint_dir)
        self.dir.mkdir(
            parents=True,
            exist_ok=True
        )

        self.file = self.dir / "checkpoint.pkl"

        self.lock = threading.RLock()
        self.stop_event = threading.Event()

        self.thread = None
        self.interval = interval
        self.config = config

        self.config_digest = hashlib.sha256(
            json.dumps(
                config,
                sort_keys=True
            ).encode()
        ).hexdigest()

        self.state = {
            "schema_version": EXPERIMENT_SCHEMA_VERSION,
            "config": config,
            "config_digest": self.config_digest,
            "results": {},
            "completed_ids": set()
        }

        self.unsaved = False

    def choose_resume_or_restart(self):

        if self.file.exists():

            choice = os.environ.get(
                "SIM_RESUME_CHOICE",
                ""
            ).strip().lower()

            while choice not in (
                "c", "continue", "1",
                "r", "restart", "2"
            ):

                choice = input(
                    "Existing Google Drive checkpoint found. "
                    "[1] Continue / "
                    "[2] Restart from beginning? "
                ).strip().lower()

            if choice in (
                "c",
                "continue",
                "1"
            ):

                try:
                    with self.file.open("rb") as f:
                        state = pickle.load(f)

                except Exception as exc:
                    raise RuntimeError(
                        "Could not read checkpoint. "
                        "Backup is preserved; resolve "
                        "or choose restart."
                    ) from exc

                if (
                    state.get("schema_version")
                    != EXPERIMENT_SCHEMA_VERSION
                    or state.get("config_digest")
                    != self.config_digest
                ):
                    raise ValueError(
                        "Checkpoint settings/code version "
                        "do not match current run. "
                        "Choose restart (2), or use "
                        "the exact original run settings."
                    )

                self.state = state

                self.state["completed_ids"] = set(
                    self.state["completed_ids"]
                )

                print(
                    f"Resumed "
                    f"{len(self.state['completed_ids'])} "
                    f"completed repetitions from "
                    f"{self.file}"
                )

            else:

                suffix = datetime.datetime.now().strftime(
                    "%Y%m%d_%H%M%S"
                )

                archived = (
                    self.dir /
                    f"checkpoint_before_restart_{suffix}.pkl"
                )

                shutil.copy2(
                    self.file,
                    archived
                )

                print(
                    f"Old checkpoint archived: "
                    f"{archived}"
                )

                self.save(force=True)

        else:

            print(
                "No existing checkpoint: "
                "starting from the beginning."
            )

            self.save(force=True)

    def completed(self, task_id: str) -> bool:

        with self.lock:
            return task_id in self.state["completed_ids"]

    def add_rows(
        self,
        bucket: str,
        task_id: str,
        rows
    ):

        with self.lock:

            if task_id in self.state["completed_ids"]:
                return

            self.state["results"].setdefault(
                bucket,
                []
            ).extend(rows)

            self.state["completed_ids"].add(
                task_id
            )

            self.unsaved = True

    def dataframe(
        self,
        bucket: str
    ) -> pd.DataFrame:

        with self.lock:

            return pd.DataFrame(
                list(
                    self.state["results"].get(
                        bucket,
                        []
                    )
                )
            )

    def save(self, force: bool = False):

        with self.lock:

            if not self.unsaved and not force:
                return

            data = pickle.dumps(
                self.state,
                protocol=pickle.HIGHEST_PROTOCOL
            )

            temp = self.dir / "checkpoint.tmp"

            try:

                with temp.open("wb") as f:

                    f.write(data)
                    f.flush()
                    os.fsync(f.fileno())

                os.replace(
                    temp,
                    self.file
                )

                self.unsaved = False

                now = datetime.datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                )

                print(
                    f"[Checkpoint {now}] "
                    f"Saved "
                    f"{len(self.state['completed_ids'])} "
                    f"completed repetitions "
                    f"to Google Drive.",
                    flush=True
                )

            except Exception:

                self.unsaved = True
                raise

    def start_periodic_saves(self):

        def saver():

            while not self.stop_event.wait(
                self.interval
            ):

                try:
                    self.save(force=True)

                except Exception as exc:

                    print(
                        "[CHECKPOINT WARNING] "
                        f"Google Drive save failed: {exc}",
                        flush=True
                    )

        self.thread = threading.Thread(
            target=saver,
            name="google-drive-checkpoints",
            daemon=True
        )

        self.thread.start()

    def close(self):

        self.stop_event.set()

        if self.thread is not None:
            self.thread.join(timeout=5)

        self.save(force=True)


def resolve_google_drive_directory() -> Path:
    """
    Use the user's mounted Google Drive.
    """

    override = os.environ.get(
        "GOOGLE_DRIVE_CHECKPOINT_DIR",
        ""
    ).strip()

    if override:

        result = Path(
            override
        ).expanduser().resolve()

        result.mkdir(
            parents=True,
            exist_ok=True
        )

        print(
            "Using user-configured Google Drive "
            f"folder: {result}"
        )

        return result

    try:
        from google.colab import drive

    except ImportError:

        raise RuntimeError(
            "Outside Google Colab, mount/sync "
            "Google Drive first and set "
            "GOOGLE_DRIVE_CHECKPOINT_DIR to "
            "a Google Drive-synced folder."
        )

    drive.mount("/content/drive")

    target = Path(
        "/content/drive/MyDrive/"
        "Source_Embeddings_Simulations_Revision/"
        "checkpoints"
    )

    target.mkdir(
        parents=True,
        exist_ok=True
    )

    return target


# ============================================================
# Simulation 1: source complexity
# ============================================================

def generate_parameters(
    K,
    regime,
    rng
):

    alpha = np.zeros(C)

    beta = rng.normal(
        loc=0.0,
        scale=SIGMA_BETA,
        size=(C, P)
    )

    if regime == "low_rank":

        U = rng.normal(
            size=(K, R0)
        )

        V = rng.normal(
            scale=1.0 / np.sqrt(R0),
            size=(C, R0)
        )

        Q = GAMMA * U @ V.T

    elif regime == "unrestricted":

        Q = rng.normal(
            loc=0.0,
            scale=GAMMA,
            size=(K, C)
        )

    else:
        raise ValueError(
            "Unknown regime."
        )

    Q = Q - Q.mean(
        axis=0,
        keepdims=True
    )

    return alpha, beta, Q


def sample_dataset(
    K,
    n_per_source,
    alpha,
    beta,
    Q,
    rng,
    balanced=True
):

    if balanced:

        S = np.repeat(
            np.arange(K),
            n_per_source
        )

        rng.shuffle(S)

        n = len(S)

    else:

        n = n_per_source

        S = rng.integers(
            low=0,
            high=K,
            size=n
        )

    X = rng.normal(
        size=(n, P)
    )

    logits = (
        alpha[None, :]
        + X @ beta.T
        + Q[S, :]
    )

    prob = softmax_np(
        logits
    )

    Y = sample_categorical(
        prob,
        rng
    )

    return (
        X.astype(np.float32),
        S.astype(np.int64),
        Y.astype(np.int64)
    )


# ============================================================
# Multinomial model
# ============================================================

class SourceMultinomialModel(nn.Module):

    def __init__(
        self,
        K,
        C,
        P,
        mode="pooled",
        m=None
    ):

        super().__init__()

        self.K = K
        self.C = C
        self.P = P
        self.mode = mode
        self.m = m

        self.alpha = nn.Parameter(
            torch.zeros(C)
        )

        self.beta = nn.Parameter(
            0.05 * torch.randn(C, P)
        )

        if mode == "One-Hot Encoding":

            self.theta = nn.Parameter(
                0.01 * torch.randn(K, C)
            )

        elif mode == "embedding":

            if m is None:
                raise ValueError(
                    "Embedding dimension m "
                    "must be provided."
                )

            self.E = nn.Parameter(
                0.05 * torch.randn(K, m)
            )

            self.A = nn.Parameter(
                0.05 * torch.randn(C, m)
            )

        elif mode == "pooled":
            pass

        else:

            raise ValueError(
                "Mode must be pooled, "
                "One-Hot Encoding, or embedding."
            )

    def source_matrix(self):

        if self.mode == "pooled":

            return torch.zeros(
                self.K,
                self.C,
                device=self.alpha.device
            )

        if self.mode == "One-Hot Encoding":

            Q = self.theta

            return Q - Q.mean(
                dim=0,
                keepdim=True
            )

        if self.mode == "embedding":

            Q = self.E @ self.A.T

            return Q - Q.mean(
                dim=0,
                keepdim=True
            )

    def penalty(self):

        if self.mode == "pooled":

            return torch.tensor(
                0.0,
                device=self.alpha.device
            )

        if self.mode == "One-Hot Encoding":

            Q = self.source_matrix()

            return (
                Q ** 2
            ).mean()

        if self.mode == "embedding":

            return (
                self.E ** 2
            ).mean() + (
                self.A ** 2
            ).mean()

    def forward(
        self,
        X,
        S
    ):

        logits = (
            X @ self.beta.T
            + self.alpha
        )

        if self.mode == "pooled":
            return logits

        Q = self.source_matrix()

        return logits + Q[S, :]


def fit_model(
    X,
    S,
    Y,
    K,
    mode,
    m=None,
    lambda_grid=LAMBDA_GRID,
    seed=1
):

    rng = np.random.default_rng(seed)

    n = X.shape[0]

    tr_idx, val_idx = train_val_split(
        n,
        VAL_FRAC,
        rng
    )

    X_t = torch.tensor(
        X,
        device=DEVICE
    )

    S_t = torch.tensor(
        S,
        device=DEVICE
    )

    Y_t = torch.tensor(
        Y,
        device=DEVICE
    )

    X_tr = X_t[tr_idx]
    S_tr = S_t[tr_idx]
    Y_tr = Y_t[tr_idx]

    X_val = X_t[val_idx]
    S_val = S_t[val_idx]
    Y_val = Y_t[val_idx]

    best_state = None
    best_val = np.inf
    best_lam = None

    for lam in lambda_grid:

        torch.manual_seed(
            seed + int(1e6 * lam)
        )

        model = SourceMultinomialModel(
            K,
            C,
            P,
            mode=mode,
            m=m
        ).to(DEVICE)

        opt = torch.optim.Adam(
            model.parameters(),
            lr=LR
        )

        best_val_lam = np.inf
        best_state_lam = None
        bad_epochs = 0

        for epoch in range(EPOCHS):

            model.train()

            opt.zero_grad()

            logits = model(
                X_tr,
                S_tr
            )

            loss = (
                F.cross_entropy(
                    logits,
                    Y_tr
                )
                + lam * model.penalty()
            )

            loss.backward()
            opt.step()

            model.eval()

            with torch.no_grad():

                val_logits = model(
                    X_val,
                    S_val
                )

                val_loss = F.cross_entropy(
                    val_logits,
                    Y_val
                ).item()

            if val_loss < best_val_lam - 1e-5:

                best_val_lam = val_loss

                best_state_lam = copy.deepcopy(
                    model.state_dict()
                )

                bad_epochs = 0

            else:

                bad_epochs += 1

            if bad_epochs >= PATIENCE:
                break

        if best_val_lam < best_val:

            best_val = best_val_lam
            best_lam = lam
            best_state = best_state_lam

    final_model = SourceMultinomialModel(
        K,
        C,
        P,
        mode=mode,
        m=m
    ).to(DEVICE)

    final_model.load_state_dict(
        best_state
    )

    final_model.eval()

    return (
        final_model,
        best_lam,
        best_val
    )


def evaluate_model(
    model,
    X_test,
    S_test,
    Y_test,
    Q_true
):

    X_t = torch.tensor(
        X_test,
        device=DEVICE
    )

    S_t = torch.tensor(
        S_test,
        device=DEVICE
    )

    Y_t = torch.tensor(
        Y_test,
        device=DEVICE
    )

    model.eval()

    with torch.no_grad():

        logits = model(
            X_t,
            S_t
        )

        logloss = F.cross_entropy(
            logits,
            Y_t
        ).item()

        Q_hat = (
            model.source_matrix()
            .detach()
            .cpu()
            .numpy()
        )

    Q_est_canonical = (
        canonical_multiclass_source_effects(
            Q_hat
        )
    )

    Q_true_canonical = (
        canonical_multiclass_source_effects(
            Q_true
        )
    )

    see = float(
        np.mean(
            (
                Q_est_canonical
                - Q_true_canonical
            ) ** 2
        )
    )

    return logloss, see


# ============================================================
# Simulation 1: single configuration
# ============================================================

def run_single_configuration(
    K,
    ns,
    regime,
    m,
    rep_seed
):

    rng = np.random.default_rng(
        rep_seed
    )

    alpha, beta, Q = generate_parameters(
        K,
        regime,
        rng
    )

    X, S, Y = sample_dataset(
        K=K,
        n_per_source=ns,
        alpha=alpha,
        beta=beta,
        Q=Q,
        rng=rng,
        balanced=True
    )

    X_test, S_test, Y_test = sample_dataset(
        K=K,
        n_per_source=N_TEST,
        alpha=alpha,
        beta=beta,
        Q=Q,
        rng=rng,
        balanced=False
    )

    rows = []

    # Pooled model

    model, lam, _ = fit_model(
        X,
        S,
        Y,
        K,
        mode="pooled",
        m=None,
        seed=rep_seed + 11
    )

    logloss, see = evaluate_model(
        model,
        X_test,
        S_test,
        Y_test,
        Q
    )

    rows.append({
        "model": "Pooled",
        "K": K,
        "ns": ns,
        "m": np.nan,
        "regime": regime,
        "logloss": logloss,
        "see": see,
        "lambda": lam,
        "rep": rep_seed
    })

    # One-hot encoding

    model, lam, _ = fit_model(
        X,
        S,
        Y,
        K,
        mode="One-Hot Encoding",
        m=None,
        seed=rep_seed + 22
    )

    logloss, see = evaluate_model(
        model,
        X_test,
        S_test,
        Y_test,
        Q
    )

    rows.append({
        "model": "One-Hot Encoding",
        "K": K,
        "ns": ns,
        "m": np.nan,
        "regime": regime,
        "logloss": logloss,
        "see": see,
        "lambda": lam,
        "rep": rep_seed
    })

    # Embedding

    model, lam, _ = fit_model(
        X,
        S,
        Y,
        K,
        mode="embedding",
        m=m,
        seed=rep_seed + 33
    )

    logloss, see = evaluate_model(
        model,
        X_test,
        S_test,
        Y_test,
        Q
    )

    rows.append({
        "model": "Embedding",
        "K": K,
        "ns": ns,
        "m": m,
        "regime": regime,
        "logloss": logloss,
        "see": see,
        "lambda": lam,
        "rep": rep_seed
    })

    return rows


# ============================================================
# Simulation 1a: log-loss versus K
# ============================================================

def run_figure_1a_simulation(
    checkpoint=None
):

    all_rows = []

    for regime in REGIMES:

        for ns in NS_GRID:

            for K in K_GRID:

                for r in range(N_REPS):

                    task_id = (
                        f"1a_{regime}_ns{ns}"
                        f"_K{K}_r{r}"
                    )

                    if (
                        checkpoint is not None
                        and checkpoint.completed(
                            task_id
                        )
                    ):
                        continue

                    seed = (
                        100000
                        + 1000 * r
                        + 10 * K
                        + ns
                        + (
                            0
                            if regime == "low_rank"
                            else 500000
                        )
                    )

                    print(
                        "[Simulation 1a] "
                        f"regime={regime}, "
                        f"ns={ns}, "
                        f"K={K}, "
                        f"rep={r+1}/{N_REPS}",
                        flush=True
                    )

                    rows = run_single_configuration(
                        K=K,
                        ns=ns,
                        regime=regime,
                        m=4,
                        rep_seed=seed
                    )

                    if checkpoint is not None:

                        checkpoint.add_rows(
                            "1a",
                            task_id,
                            rows
                        )

                    else:

                        all_rows.extend(
                            rows
                        )

    df = (
        checkpoint.dataframe("1a")
        if checkpoint is not None
        else pd.DataFrame(all_rows)
    )

    df.to_csv(
        os.path.join(
            OUTDIR,
            "source_complexity_logloss_results.csv"
        ),
        index=False
    )

    return df


# ============================================================
# Simulation 1b: embedding dimension
# ============================================================

def run_figure_1b_simulation(
    checkpoint=None
):

    all_rows = []

    K_FIXED = 50

    for regime in REGIMES:

        for ns in NS_GRID:

            for m in M_GRID:

                for r in range(N_REPS):

                    task_id = (
                        f"1b_{regime}_ns{ns}"
                        f"_m{m}_r{r}"
                    )

                    if (
                        checkpoint is not None
                        and checkpoint.completed(
                            task_id
                        )
                    ):
                        continue

                    seed = (
                        200000
                        + 1000 * r
                        + 100 * m
                        + ns
                        + (
                            0
                            if regime == "low_rank"
                            else 500000
                        )
                    )

                    print(
                        "[Simulation 1b] "
                        f"regime={regime}, "
                        f"ns={ns}, "
                        f"m={m}, "
                        f"rep={r+1}/{N_REPS}",
                        flush=True
                    )

                    rng = np.random.default_rng(
                        seed
                    )

                    alpha, beta, Q = generate_parameters(
                        K_FIXED,
                        regime,
                        rng
                    )

                    X, S, Y = sample_dataset(
                        K_FIXED,
                        ns,
                        alpha,
                        beta,
                        Q,
                        rng,
                        balanced=True
                    )

                    X_test, S_test, Y_test = sample_dataset(
                        K_FIXED,
                        N_TEST,
                        alpha,
                        beta,
                        Q,
                        rng,
                        balanced=False
                    )

                    model, lam, _ = fit_model(
                        X,
                        S,
                        Y,
                        K_FIXED,
                        mode="embedding",
                        m=m,
                        seed=seed + 33
                    )

                    logloss, see = evaluate_model(
                        model,
                        X_test,
                        S_test,
                        Y_test,
                        Q
                    )

                    row = {
                        "model": "Embedding",
                        "K": K_FIXED,
                        "ns": ns,
                        "m": m,
                        "regime": regime,
                        "logloss": logloss,
                        "see": see,
                        "lambda": lam,
                        "rep": seed
                    }

                    if checkpoint is not None:

                        checkpoint.add_rows(
                            "1b",
                            task_id,
                            [row]
                        )

                    else:

                        all_rows.append(
                            row
                        )

    df = (
        checkpoint.dataframe("1b")
        if checkpoint is not None
        else pd.DataFrame(all_rows)
    )

    df.to_csv(
        os.path.join(
            OUTDIR,
            "source_complexity_embedding_dim_results.csv"
        ),
        index=False
    )

    return df


# ============================================================
# Summary helper
# ============================================================

def summarize(
    df,
    group_cols,
    metric
):

    out = (
        df.groupby(
            group_cols
        )[metric]
        .agg([
            "mean",
            "std",
            "count"
        ])
        .reset_index()
    )

    out["se"] = (
        out["std"]
        / np.sqrt(
            out["count"]
        )
    )

    return out


def regime_label(regime):

    if regime == "low_rank":
        return "Low-rank source effects"

    return "Unrestricted source effects"


# ============================================================
# UPDATED FIGURE 1
#
# ONLY TWO PANELS:
#     Low-rank, n_s = 40
#     Low-rank, n_s = 200
#
# Output filename remains unchanged.
# ============================================================

def plot_figure_1a(df):

    """
    Figure 1: Test log-loss versus K.

    ONLY the first two panels are displayed:

        (a) Low-rank source effects, n_s=40
        (b) Low-rank source effects, n_s=200

    The unrestricted simulations are preserved
    in the CSV but excluded from this figure.
    """

    summary = summarize(
        df,
        group_cols=[
            "regime",
            "ns",
            "K",
            "model"
        ],
        metric="logloss"
    )

    colors = {
        "Pooled": "#4D4D4D",
        "One-Hot Encoding": "#0072B2",
        "Embedding": "#D55E00"
    }

    # CHANGED: original 2x2 grid -> 1x2 grid
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10.5, 4.2),
        sharex=True,
        sharey=True
    )

    # CHANGED: show only low-rank settings
    panel_order = [
        ("low_rank", 40),
        ("low_rank", 200)
    ]

    for ax, (regime, ns) in zip(
        axes,
        panel_order
    ):

        sub = summary[
            (summary["regime"] == regime)
            & (summary["ns"] == ns)
        ]

        for model in [
            "Pooled",
            "One-Hot Encoding",
            "Embedding"
        ]:

            sm = sub[
                sub["model"] == model
            ].sort_values("K")

            ax.errorbar(
                sm["K"],
                sm["mean"],
                yerr=sm["se"],
                marker="o",
                linewidth=2,
                capsize=3,
                markersize=6,
                label=model,
                color=colors[model]
            )

        ax.set_title(
            f"Low-rank source effects, $n_s={ns}$"
        )

        ax.set_xlabel(
            "Number of sources $K$"
        )

        ax.set_xticks(
            K_GRID
        )

        ax.grid(
            alpha=0.25
        )

    axes[0].set_ylabel(
        "Test log-loss"
    )

    # Common legend
    handles, labels = (
        axes[0].get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=3,
        frameon=False
    )

    fig.tight_layout(
        rect=[0, 0, 1, 0.88]
    )

    path = os.path.join(
        OUTDIR,
        "figure_source_complexity_logloss.png"
    )

    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(
        f"Saved {path}"
    )


# ============================================================
# Figure 1b: embedding dimension
# ============================================================

def plot_figure_1b(df):

    s_log = summarize(
        df,
        group_cols=[
            "regime",
            "ns",
            "m"
        ],
        metric="logloss"
    )

    s_see = summarize(
        df,
        group_cols=[
            "regime",
            "ns",
            "m"
        ],
        metric="see"
    )

    colors = {
        40: "#D55E00",
        200: "#0072B2"
    }

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(10.5, 7.2),
        sharex=True
    )

    for row, regime in enumerate(
        REGIMES
    ):

        # Log-loss

        ax = axes[row, 0]

        sub = s_log[
            s_log["regime"] == regime
        ]

        for ns in NS_GRID:

            sm = sub[
                sub["ns"] == ns
            ].sort_values("m")

            ax.errorbar(
                sm["m"],
                sm["mean"],
                yerr=sm["se"],
                marker="o",
                linewidth=2,
                capsize=3,
                label=f"$n_s={ns}$",
                color=colors[ns]
            )

        ax.axvline(
            R0,
            color="black",
            linestyle="--",
            linewidth=1
        )

        ax.set_title(
            f"{regime_label(regime)}: "
            "test log-loss"
        )

        ax.set_ylabel(
            "Test log-loss"
        )

        ax.grid(
            alpha=0.25
        )

        # Centered source-effect error

        ax = axes[row, 1]

        sub = s_see[
            s_see["regime"] == regime
        ]

        for ns in NS_GRID:

            sm = sub[
                sub["ns"] == ns
            ].sort_values("m")

            ax.errorbar(
                sm["m"],
                sm["mean"],
                yerr=sm["se"],
                marker="o",
                linewidth=2,
                capsize=3,
                label=f"$n_s={ns}$",
                color=colors[ns]
            )

        ax.axvline(
            R0,
            color="black",
            linestyle="--",
            linewidth=1
        )

        ax.set_title(
            f"{regime_label(regime)}: "
            "centered source-effect error"
        )

        ax.set_ylabel(
            "Centered source-effect MSE"
        )

        ax.grid(
            alpha=0.25
        )

    for ax in axes[-1, :]:

        ax.set_xlabel(
            "Embedding dimension $m$"
        )

        ax.set_xticks(
            M_GRID
        )

    handles, labels = (
        axes[0, 0].get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=2,
        frameon=False
    )

    fig.tight_layout(
        rect=[0, 0, 1, 0.94]
    )

    path = os.path.join(
        OUTDIR,
        "figure_source_complexity_embedding_dim.png"
    )

    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(
        f"Saved {path}"
    )


# ============================================================
# Simulation 2: covariance DGP
# ============================================================

@dataclass
class CovarianceDGP:

    n: int = 6000
    p: int = 4
    K: int = 4
    C: int = 3
    m: int = 3
    delta: float = 1.0
    noise_sd: float = 1.0
    seed: int = 456


def generate_covariance_data(
    dgp: CovarianceDGP
) -> Tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray
]:

    rng = np.random.default_rng(
        dgp.seed
    )

    Y = rng.choice(
        dgp.C,
        size=dgp.n
    )

    S = rng.choice(
        dgp.K,
        size=dgp.n
    )

    U = rng.normal(
        0,
        1.0,
        size=(
            dgp.C,
            dgp.p
        )
    )

    E = rng.normal(
        0,
        1.0,
        size=(
            dgp.K,
            dgp.m
        )
    )

    G = rng.normal(
        0,
        1.0 / np.sqrt(dgp.m),
        size=(
            dgp.m,
            dgp.p
        )
    )

    eps = rng.normal(
        0,
        dgp.noise_sd,
        size=(
            dgp.n,
            dgp.p
        )
    )

    H = (
        U[Y]
        + dgp.delta * (E[S] @ G)
        + eps
    )

    return H, Y, S


def compute_covariance_quantities(
    H: np.ndarray,
    Y: np.ndarray,
    S: np.ndarray
) -> Dict[str, float]:

    pooled_trace = float(
        np.trace(
            pooled_within_class_covariance(
                H,
                Y
            )
        )
    )

    n = len(Y)

    classes = np.unique(Y)
    sources = np.unique(S)

    p = H.shape[1]

    sigma_cond = np.zeros(
        (p, p)
    )

    for c in classes:

        for s in sources:

            idx = (
                (Y == c)
                & (S == s)
            )

            hcs = H[idx]

            if hcs.shape[0] <= 1:
                continue

            weight = (
                hcs.shape[0] / n
            )

            centered = (
                hcs
                - hcs.mean(
                    axis=0,
                    keepdims=True
                )
            )

            cov_cs = (
                centered.T @ centered
                / hcs.shape[0]
            )

            sigma_cond += (
                weight * cov_cs
            )

    cond_trace = float(
        np.trace(
            sigma_cond
        )
    )

    return {
        "pooled_within_trace":
            pooled_trace,

        "source_conditioned_within_trace":
            cond_trace,

        "source_induced_inflation":
            pooled_trace - cond_trace
    }


# ============================================================
# Simulation 3: regression DGP
# ============================================================

@dataclass
class OVBdgp:

    n: int = 5000
    p: int = 5
    K: int = 6
    m: int = 4
    delta: float = 1.0
    beta_scale: float = 1.0
    x_s_dependence: float = 0.7
    noise_sd: float = 1.0
    seed: int = 789


def generate_ovb_data(
    dgp: OVBdgp
) -> Tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray
]:

    rng = np.random.default_rng(
        dgp.seed
    )

    S = rng.choice(
        dgp.K,
        size=dgp.n
    )

    E0 = rng.normal(
        0,
        1.0,
        size=(
            dgp.K,
            dgp.m
        )
    )

    Bz = rng.normal(
        0,
        (
            dgp.x_s_dependence
            / np.sqrt(dgp.m)
        ),
        size=(
            dgp.m,
            dgp.p
        )
    )

    zeta = rng.normal(
        0,
        1.0,
        size=(
            dgp.n,
            dgp.p
        )
    )

    Z = (
        zeta
        + E0[S] @ Bz
    )

    beta0 = rng.normal(
        0,
        dgp.beta_scale,
        size=dgp.p
    )

    a0 = rng.normal(
        0,
        1.0,
        size=dgp.m
    )

    eps = rng.normal(
        0,
        dgp.noise_sd,
        size=dgp.n
    )

    q0 = (
        dgp.delta
        * (E0 @ a0)
    )

    T = (
        Z @ beta0
        + q0[S]
        + eps
    )

    return (
        Z,
        S,
        T,
        beta0,
        q0
    )


# ============================================================
# Jointly trainable source-aware models
# ============================================================

class SourceAwareMulticlass(nn.Module):

    """
    eta_c(X,S) =
        linear_X_c(X) + <e_S,a_c>

    Both E and the loading matrix A are trainable.
    """

    def __init__(
        self,
        p: int,
        K: int,
        C: int,
        m: int
    ):

        super().__init__()

        self.linear = nn.Linear(
            p,
            C
        )

        self.embedding = nn.Embedding(
            K,
            m
        )

        self.A = nn.Parameter(
            torch.zeros(m, C)
        )

        nn.init.normal_(
            self.linear.weight,
            mean=0.0,
            std=0.05
        )

        nn.init.zeros_(
            self.linear.bias
        )

        nn.init.normal_(
            self.embedding.weight,
            mean=0.0,
            std=0.1
        )

        nn.init.normal_(
            self.A,
            mean=0.0,
            std=0.1
        )

    def forward(
        self,
        x: torch.Tensor,
        s: torch.Tensor
    ) -> torch.Tensor:

        base = self.linear(
            x
        )

        e = self.embedding(
            s
        )

        source_term = (
            e @ self.A
        )

        return (
            base + source_term
        )


class SourceAwareLinear(nn.Module):

    """
    eta(Z,S) =
        beta^T Z + <e_S,a>

    beta, E, and a are jointly trainable.
    """

    def __init__(
        self,
        p: int,
        K: int,
        m: int
    ):

        super().__init__()

        self.linear = nn.Linear(
            p,
            1
        )

        self.embedding = nn.Embedding(
            K,
            m
        )

        self.a = nn.Parameter(
            torch.zeros(m, 1)
        )

        nn.init.normal_(
            self.linear.weight,
            mean=0.0,
            std=0.05
        )

        nn.init.zeros_(
            self.linear.bias
        )

        nn.init.normal_(
            self.embedding.weight,
            mean=0.0,
            std=0.1
        )

        nn.init.normal_(
            self.a,
            mean=0.0,
            std=0.1
        )

    def forward(
        self,
        z: torch.Tensor,
        s: torch.Tensor
    ) -> torch.Tensor:

        base = self.linear(
            z
        )

        e = self.embedding(
            s
        )

        source_term = (
            e @ self.a
        )

        return (
            base + source_term
        )


# ============================================================
# Fit source-aware multiclass model
# ============================================================

def fit_source_aware_multiclass(
    X_train: np.ndarray,
    S_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    S_test: np.ndarray,
    p: int,
    K: int,
    C: int,
    m: int,
    seed: int,
    lr: float = 1e-2,
    weight_decay: float = 1e-4,
    batch_size: int = 256,
    epochs: int = 50
) -> Tuple[np.ndarray, np.ndarray]:

    set_all_seeds(
        seed
    )

    model = SourceAwareMulticlass(
        p=p,
        K=K,
        C=C,
        m=m
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    dataset = TensorDataset(
        torch.tensor(
            X_train,
            dtype=TORCH_DTYPE
        ),
        torch.tensor(
            S_train,
            dtype=torch.long
        ),
        torch.tensor(
            y_train,
            dtype=torch.long
        )
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    model.train()

    for _ in range(epochs):

        for xb, sb, yb in loader:

            xb = xb.to(DEVICE)
            sb = sb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(
                xb,
                sb
            )

            loss = criterion(
                logits,
                yb
            )

            loss.backward()
            optimizer.step()

    model.eval()

    with torch.no_grad():

        xt = torch.tensor(
            X_test,
            dtype=TORCH_DTYPE,
            device=DEVICE
        )

        st = torch.tensor(
            S_test,
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(
            xt,
            st
        ).cpu().numpy()

        probs = softmax_np(
            logits
        )

        pred = probs.argmax(
            axis=1
        )

    return pred, probs


# ============================================================
# Fit source-aware linear regression
# ============================================================

def fit_source_aware_linear(
    Z_train: np.ndarray,
    S_train: np.ndarray,
    T_train: np.ndarray,
    Z_test: np.ndarray,
    S_test: np.ndarray,
    p: int,
    K: int,
    m: int,
    seed: int,
    lr: float = 1e-2,
    weight_decay: float = 1e-4,
    batch_size: int = 256,
    epochs: int = 200
) -> Tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray
]:

    set_all_seeds(
        seed
    )

    model = SourceAwareLinear(
        p=p,
        K=K,
        m=m
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    criterion = nn.MSELoss()

    dataset = TensorDataset(
        torch.tensor(
            Z_train,
            dtype=TORCH_DTYPE
        ),
        torch.tensor(
            S_train,
            dtype=torch.long
        ),
        torch.tensor(
            T_train.reshape(-1, 1),
            dtype=TORCH_DTYPE
        )
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    model.train()

    for _ in range(epochs):

        for zb, sb, tb in loader:

            zb = zb.to(DEVICE)
            sb = sb.to(DEVICE)
            tb = tb.to(DEVICE)

            optimizer.zero_grad()

            pred = model(
                zb,
                sb
            )

            loss = criterion(
                pred,
                tb
            )

            loss.backward()
            optimizer.step()

    model.eval()

    with torch.no_grad():

        zt = torch.tensor(
            Z_test,
            dtype=TORCH_DTYPE,
            device=DEVICE
        )

        st = torch.tensor(
            S_test,
            dtype=torch.long,
            device=DEVICE
        )

        pred_test = model(
            zt,
            st
        ).cpu().numpy().reshape(-1)

        beta_hat = (
            model.linear.weight
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

        q_hat = (
            model.embedding.weight @ model.a
        ).detach().cpu().numpy().reshape(-1)

    return (
        pred_test,
        beta_hat,
        q_hat
    )


# ============================================================
# Classification experiment
# ============================================================

def run_classification_experiment(
    n_reps: int = 100,
    delta_grid: List[float] = (
        0.0,
        0.25,
        0.5,
        1.0,
        1.5
    ),
    base_seed: int = 1000,
    checkpoint=None
) -> pd.DataFrame:

    rows = []

    K1, m1, p1, C1 = (
        5,
        2,
        6,
        3
    )

    for delta in delta_grid:

        for rep in range(n_reps):

            task_id = (
                f"1c_class_delta{delta:g}"
                f"_rep{rep}"
            )

            if (
                checkpoint is not None
                and checkpoint.completed(
                    task_id
                )
            ):
                continue

            seed = (
                base_seed
                + 10000 * int(100 * delta)
                + rep
            )

            dgp = ClassificationDGP(
                n=4000,
                p=p1,
                K=K1,
                C=C1,
                m=m1,
                delta=delta,
                seed=seed
            )

            X, S, Y = (
                generate_source_aware_classification(
                    dgp
                )
            )

            (
                X_train,
                X_test,
                S_train,
                S_test,
                Y_train,
                Y_test
            ) = train_test_split(
                X,
                S,
                Y,
                test_size=0.4,
                random_state=seed,
                stratify=Y
            )

            pooled = LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                random_state=seed
            )

            pooled.fit(
                X_train,
                Y_train
            )

            pred_x = pooled.predict(
                X_test
            )

            prob_x = pooled.predict_proba(
                X_test
            )

            pred_xs, prob_xs = (
                fit_source_aware_multiclass(
                    X_train,
                    S_train,
                    Y_train,
                    X_test,
                    S_test,
                    p=p1,
                    K=K1,
                    C=C1,
                    m=m1,
                    seed=seed
                )
            )

            err_x = (
                1 - accuracy_score(
                    Y_test,
                    pred_x
                )
            )

            err_xs = (
                1 - accuracy_score(
                    Y_test,
                    pred_xs
                )
            )

            row = {
                "rep": rep,
                "delta": delta,
                "K": K1,
                "m": m1,
                "err_x": err_x,
                "err_xs": err_xs,
                "logloss_x": log_loss(
                    Y_test,
                    prob_x
                ),
                "logloss_xs": log_loss(
                    Y_test,
                    prob_xs
                ),
                "improvement_error":
                    err_x - err_xs,
                "improvement_logloss":
                    log_loss(Y_test, prob_x)
                    - log_loss(Y_test, prob_xs)
            }

            if checkpoint is not None:

                checkpoint.add_rows(
                    "cls",
                    task_id,
                    [row]
                )

            else:

                rows.append(
                    row
                )

    return (
        checkpoint.dataframe("cls")
        if checkpoint is not None
        else pd.DataFrame(rows)
    )


# ============================================================
# Covariance experiment
# ============================================================

def run_covariance_experiment(
    n_reps: int = 100,
    delta_grid: List[float] = (
        0.0,
        0.25,
        0.5,
        1.0,
        1.5
    ),
    base_seed: int = 2000,
    checkpoint=None
) -> pd.DataFrame:

    rows = []

    K2, m2 = (
        4,
        3
    )

    for delta in delta_grid:

        for rep in range(n_reps):

            task_id = (
                f"2_cov_delta{delta:g}"
                f"_rep{rep}"
            )

            if (
                checkpoint is not None
                and checkpoint.completed(
                    task_id
                )
            ):
                continue

            seed = (
                base_seed
                + 10000 * int(100 * delta)
                + rep
            )

            dgp = CovarianceDGP(
                n=6000,
                p=4,
                K=K2,
                C=3,
                m=m2,
                delta=delta,
                seed=seed
            )

            H, Y, S = (
                generate_covariance_data(
                    dgp
                )
            )

            out = compute_covariance_quantities(
                H,
                Y,
                S
            )

            out.update({
                "rep": rep,
                "delta": delta,
                "K": K2,
                "m": m2
            })

            if checkpoint is not None:

                checkpoint.add_rows(
                    "cov",
                    task_id,
                    [out]
                )

            else:

                rows.append(
                    out
                )

    return (
        checkpoint.dataframe("cov")
        if checkpoint is not None
        else pd.DataFrame(rows)
    )


# ============================================================
# Omitted-source regression experiment
# ============================================================

def run_ovb_experiment(
    n_reps: int = 150,
    delta_grid: List[float] = (
        0.0,
        0.5,
        1.0,
        1.5
    ),
    base_seed: int = 3000,
    checkpoint=None
) -> pd.DataFrame:

    rows = []

    K3, m3, p3 = (
        6,
        4,
        5
    )

    for delta in delta_grid:

        for rep in range(n_reps):

            task_id = (
                f"3_ovb_delta{delta:g}"
                f"_rep{rep}"
            )

            if (
                checkpoint is not None
                and checkpoint.completed(
                    task_id
                )
            ):
                continue

            seed = (
                base_seed
                + 10000 * int(100 * delta)
                + rep
            )

            dgp = OVBdgp(
                n=5000,
                p=p3,
                K=K3,
                m=m3,
                delta=delta,
                seed=seed
            )

            Z, S, T, beta0, q0 = (
                generate_ovb_data(
                    dgp
                )
            )

            omit = LinearRegression()

            omit.fit(
                Z,
                T
            )

            beta_tilde = omit.coef_

            resid_omit = (
                T - omit.predict(Z)
            )

            (
                pred_full,
                beta_full,
                q_hat
            ) = fit_source_aware_linear(
                Z,
                S,
                T,
                Z,
                S,
                p=p3,
                K=K3,
                m=m3,
                seed=seed
            )

            resid_full = (
                T - pred_full
            )

            # Center source effects:
            # only contrasts are identifiable
            # when an intercept is included.

            q_true_centered = (
                q0 - q0.mean()
            )

            q_hat_centered = (
                q_hat - q_hat.mean()
            )

            row = {

                "rep": rep,
                "delta": delta,
                "K": K3,
                "m": m3,

                "beta_error_norm_omit":
                    float(
                        np.linalg.norm(
                            beta_tilde - beta0
                        )
                    ),

                "beta_error_norm_full":
                    float(
                        np.linalg.norm(
                            beta_full - beta0
                        )
                    ),

                "beta_bias_norm_omit":
                    float(
                        np.linalg.norm(
                            beta_tilde - beta0
                        )
                    ),

                "beta_bias_norm_full":
                    float(
                        np.linalg.norm(
                            beta_full - beta0
                        )
                    ),

                "resid_var_omit":
                    float(
                        np.var(
                            resid_omit
                        )
                    ),

                "resid_var_full":
                    float(
                        np.var(
                            resid_full
                        )
                    ),

                "resid_var_inflation":
                    float(
                        np.var(resid_omit)
                        - np.var(resid_full)
                    ),

                "see_full":
                    float(
                        np.mean(
                            (
                                q_hat_centered
                                - q_true_centered
                            ) ** 2
                        )
                    ),

                "see_omit":
                    float(
                        np.mean(
                            q_true_centered ** 2
                        )
                    )
            }

            for s in range(K3):

                row[
                    f"q_true_source_{s+1}"
                ] = float(
                    q_true_centered[s]
                )

                row[
                    f"q_hat_source_{s+1}"
                ] = float(
                    q_hat_centered[s]
                )

            if checkpoint is not None:

                checkpoint.add_rows(
                    "ovb",
                    task_id,
                    [row]
                )

            else:

                rows.append(
                    row
                )

    return (
        checkpoint.dataframe("ovb")
        if checkpoint is not None
        else pd.DataFrame(rows)
    )


# ============================================================
# Plot classification figures
# ============================================================

def plot_prediction_figure(
    df_cls: pd.DataFrame
) -> None:

    summ = summarize_with_ci(
        df_cls,
        "delta",
        [
            "err_x",
            "err_xs",
            "improvement_error"
        ]
    )

    x = summ["delta"].values

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.errorbar(
        x,
        summ["err_x_mean"],
        yerr=summ["err_x_ci"],
        marker="o",
        linewidth=2,
        capsize=4,
        label="Source-agnostic"
    )

    ax.errorbar(
        x,
        summ["err_xs_mean"],
        yerr=summ["err_xs_ci"],
        marker="s",
        linewidth=2,
        capsize=4,
        label="Source-aware"
    )

    ax.set_xlabel(
        "Source-shift magnitude"
    )

    ax.set_ylabel(
        "Test classification error"
    )

    ax.set_title(
        "Prediction error under increasing "
        "source heterogeneity"
    )

    ax.legend(
        frameon=False
    )

    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTDIR,
            "figure2.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    # Prediction gain

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.errorbar(
        x,
        summ["improvement_error_mean"],
        yerr=summ["improvement_error_ci"],
        marker="o",
        linewidth=2,
        capsize=4
    )

    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "Source-shift magnitude"
    )

    ax.set_ylabel(
        "Error reduction from source awareness"
    )

    ax.set_title(
        "Performance gain from incorporating "
        "source information"
    )

    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTDIR,
            "figure3.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# Plot covariance figures
# ============================================================

def plot_covariance_figure(
    df_cov: pd.DataFrame
) -> None:

    summ = summarize_with_ci(
        df_cov,
        "delta",
        [
            "pooled_within_trace",
            "source_conditioned_within_trace",
            "source_induced_inflation"
        ]
    )

    x = summ["delta"].values

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.errorbar(
        x,
        summ["pooled_within_trace_mean"],
        yerr=summ["pooled_within_trace_ci"],
        marker="o",
        linewidth=2,
        capsize=4,
        label="Ignoring source"
    )

    ax.errorbar(
        x,
        summ["source_conditioned_within_trace_mean"],
        yerr=summ["source_conditioned_within_trace_ci"],
        marker="s",
        linewidth=2,
        capsize=4,
        label="Conditioning on source"
    )

    ax.set_xlabel(
        "Source-shift magnitude"
    )

    ax.set_ylabel(
        "Trace of within-class covariance"
    )

    ax.set_title(
        "Within-class covariance inflation "
        "from source mixing"
    )

    ax.legend(
        frameon=False
    )

    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTDIR,
            "figure4.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    # Source-induced inflation

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.errorbar(
        x,
        summ["source_induced_inflation_mean"],
        yerr=summ["source_induced_inflation_ci"],
        marker="o",
        linewidth=2,
        capsize=4
    )

    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "Source-shift magnitude"
    )

    ax.set_ylabel(
        "Source-induced covariance inflation"
    )

    ax.set_title(
        "Additional within-class variability "
        "due to source heterogeneity"
    )

    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTDIR,
            "figure1.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# Plot omitted-source regression figures
# ============================================================

def plot_ovb_figure(
    df_ovb: pd.DataFrame
):

    summ = summarize_with_ci(
        df_ovb,
        "delta",
        [
            "beta_error_norm_omit",
            "beta_error_norm_full",
            "resid_var_inflation",
            "see_omit",
            "see_full"
        ]
    )

    x = summ["delta"].values

    # Coefficient error

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.errorbar(
        x,
        summ["beta_error_norm_omit_mean"],
        yerr=summ["beta_error_norm_omit_ci"],
        marker="o",
        linewidth=2,
        capsize=4,
        label="Omitting source"
    )

    ax.errorbar(
        x,
        summ["beta_error_norm_full_mean"],
        yerr=summ["beta_error_norm_full_ci"],
        marker="s",
        linewidth=2,
        capsize=4,
        label="Source-aware"
    )

    ax.set_xlabel(
        "Magnitude of source effect"
    )

    ax.set_ylabel(
        r"$\|\hat{\boldsymbol{\beta}}"
        r"-\boldsymbol{\beta}_0\|_2$"
    )

    ax.set_title(
        "Coefficient estimation error "
        "under source omission"
    )

    ax.legend(
        frameon=False
    )

    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTDIR,
            "figure5.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    # Residual inflation

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.errorbar(
        x,
        summ["resid_var_inflation_mean"],
        yerr=summ["resid_var_inflation_ci"],
        marker="o",
        linewidth=2,
        capsize=4
    )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "Magnitude of source effect"
    )

    ax.set_ylabel(
        "Residual variance inflation"
    )

    ax.set_title(
        "Residual inflation induced "
        "by omitting source information"
    )

    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTDIR,
            "figure6.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    # Source-effect recovery

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.errorbar(
        x,
        summ["see_omit_mean"],
        yerr=summ["see_omit_ci"],
        marker="o",
        linewidth=2,
        capsize=4,
        label="Omitting source (zero contrasts)"
    )

    ax.errorbar(
        x,
        summ["see_full_mean"],
        yerr=summ["see_full_ci"],
        marker="s",
        linewidth=2,
        capsize=4,
        label="Source-aware fitted effects"
    )

    ax.set_xlabel(
        "Magnitude of source effect"
    )

    ax.set_ylabel(
        "Centered source-effect MSE"
    )

    ax.set_title(
        "Recovery of true source effects q(s)"
    )

    ax.legend(
        frameon=False
    )

    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTDIR,
            "figure7_source_effect_recovery.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    return summ


# ============================================================
# Main runner
# ============================================================

def run_all_experiments():

    global OUTDIR

    drive_ckpt_dir = (
        resolve_google_drive_directory()
    )

    OUTDIR = str(
        drive_ckpt_dir.parent / "outputs"
    )

    Path(OUTDIR).mkdir(
        parents=True,
        exist_ok=True
    )

    config = {

        "schema_version":
            EXPERIMENT_SCHEMA_VERSION,

        "N_REPS": N_REPS,
        "C": C,
        "P": P,
        "R0": R0,
        "GAMMA": GAMMA,
        "SIGMA_BETA": SIGMA_BETA,

        "K_GRID": K_GRID,
        "M_GRID": M_GRID,
        "NS_GRID": NS_GRID,
        "REGIMES": REGIMES,

        "N_TEST": N_TEST,
        "LAMBDA_GRID": LAMBDA_GRID,

        "EPOCHS": EPOCHS,
        "LR": LR,
        "PATIENCE": PATIENCE,
        "VAL_FRAC": VAL_FRAC,

        "cls_n_reps": 100,
        "cov_n_reps": 100,
        "ovb_n_reps": 150,

        "cls_delta": [
            0.0,
            0.25,
            0.5,
            1.0,
            1.5
        ],

        "cov_delta": [
            0.0,
            0.25,
            0.5,
            1.0,
            1.5
        ],

        "ovb_delta": [
            0.0,
            0.5,
            1.0,
            1.5
        ]
    }

    cp = CheckpointManager(
        drive_ckpt_dir,
        config,
        CHECKPOINT_INTERVAL_SECONDS
    )

    cp.choose_resume_or_restart()

    cp.start_periodic_saves()

    print(
        "Outputs:",
        OUTDIR
    )

    # Default behavior remains unchanged:
    # run revised simulations 1a, 1b, and OVB.
    #
    # Figure 1a is now plotted with only
    # the two low-rank panels.
    #
    # Other phases can be requested using:
    #
    # SIM_RUN_ONLY=1a,1b,cls,cov,ovb

    requested = os.environ.get(
        "SIM_RUN_ONLY",
        "1a,1b,ovb"
    )

    phases = [
        s.strip()
        for s in requested.split(",")
        if s.strip()
    ]

    try:

        for phase in phases:

            print(
                f"== START/RESUME {phase} ==",
                flush=True
            )

            if phase == "1a":

                df = run_figure_1a_simulation(
                    checkpoint=cp
                )

                # Updated two-panel Figure 1
                plot_figure_1a(df)

            elif phase == "1b":

                df = run_figure_1b_simulation(
                    checkpoint=cp
                )

                plot_figure_1b(df)

            elif phase == "cls":

                df = run_classification_experiment(
                    checkpoint=cp
                )

                df.to_csv(
                    os.path.join(
                        OUTDIR,
                        "section_3_1_classification_results.csv"
                    ),
                    index=False
                )

                plot_prediction_figure(df)

            elif phase == "cov":

                df = run_covariance_experiment(
                    checkpoint=cp
                )

                df.to_csv(
                    os.path.join(
                        OUTDIR,
                        "section_3_1_covariance_results.csv"
                    ),
                    index=False
                )

                plot_covariance_figure(df)

            elif phase == "ovb":

                df = run_ovb_experiment(
                    checkpoint=cp
                )

                df.to_csv(
                    os.path.join(
                        OUTDIR,
                        "section_3_1_ovb_results.csv"
                    ),
                    index=False
                )

                summ = plot_ovb_figure(
                    df
                )

                summ.to_csv(
                    os.path.join(
                        OUTDIR,
                        "section_3_1_ovb_source_effect_summary.csv"
                    ),
                    index=False
                )

                by_source = []

                for _, rec in df.iterrows():

                    for s in range(
                        1,
                        int(rec["K"]) + 1
                    ):

                        by_source.append({

                            "rep":
                                int(rec["rep"]),

                            "delta":
                                rec["delta"],

                            "source":
                                s,

                            "q_true_centered":
                                rec[
                                    f"q_true_source_{s}"
                                ],

                            "q_hat_centered":
                                rec[
                                    f"q_hat_source_{s}"
                                ]
                        })

                pd.DataFrame(
                    by_source
                ).to_csv(
                    os.path.join(
                        OUTDIR,
                        "section_3_1_ovb_source_effect_by_source.csv"
                    ),
                    index=False
                )

            else:

                raise ValueError(
                    f"Unknown phase {phase!r}; "
                    "use 1a,1b,cls,cov,ovb"
                )

            cp.save(
                force=True
            )

            print(
                f"== FINISHED {phase}: "
                "outputs saved to Google Drive ==",
                flush=True
            )

    except BaseException:

        print(
            "Stopped/failed. Saving completed "
            "work to Google Drive before exit.",
            flush=True
        )

        raise

    finally:

        cp.close()


# ============================================================
# Execute
# ============================================================

if __name__ == "__main__":

    run_all_experiments()


Mounted at /content/drive
No existing checkpoint: starting from the beginning.
[Checkpoint 2026-10-28 12:38:48] Saved 0 completed repetitions to Google Drive.
Outputs: /content/drive/MyDrive/Source_Embeddings_Simulations_Revision/outputs
== START/RESUME 1a ==
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=1/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=2/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=3/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=4/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=5/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=6/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=7/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=8/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=9/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=10/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=11/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=12/100
[Simulation 1a] regime=low_rank, ns=40, K=10, rep=13/100
[Simulat